# `metrics.py`


## Function summary

| Function | Purpose | Key inputs | Input structure | Output | Output structure |
|---|---|---|---|---|---|
| `compute_spc` | Recall probability at each serial position | `recall_sims`, `N` | `recall_sims`: `(N, n_sims)` int array; entries are 1-based serial positions, `0` = padding. `N`: int list length | SPC vector | `(N,)` float array; one recall probability per serial position |
| `compute_pfr` | Probability that each serial position is recalled first | `recall_sims`, `N` | Same `recall_sims` structure as above; `N`: int | PFR vector | `(N,)` float array; one first-recall probability per serial position |
| `compute_lag_crp` | Opportunity-corrected lag-CRP across the whole list | `recall_sims`, `N` | `recall_sims`: `(N, n_sims)` int array; `N`: int | Lag axis and CRP | Tuple: `lag_vals = (2N−1,)` int array, `crp = (2N−1,)` float array |
| `recall_accuracy` | Mean fraction of items recalled per trial | `recall_sims`, `N`, `unique` | `recall_sims`: `(N, n_sims)` int array; `N`: int; `unique`: bool controlling whether repeats are collapsed | Mean recall accuracy | Scalar float, or `NaN` if input is empty |
| `position_conditional_crp` | Opportunity-corrected transition probability for one exact start→target relation | `recall_sims`, `N`, `start_pos`, `target_lag`, `return_counts` | `start_pos`: 1-based int; `target_lag`: signed int offset from `start_pos`; `return_counts`: bool | Exact transition probability, optionally with raw counts | Either scalar float, or `TransitionResult(prob, num, den)` |
| `boundary_transition` | Boundary-relative transition probability pooled across all boundaries | `recall_sims`, `N`, `start_offset`, `target_offset`, `boundary_positions`, `return_counts` | `start_offset` and `target_offset` are relative to each boundary `j`; `boundary_positions`: list of 1-based ints | Pooled boundary-relative transition probability | Either scalar float, or `TransitionResult(prob, num, den)` pooled across boundaries |
| `local_spc` | SPC in a local window around one chosen serial position | `recall_sims`, `N`, `center`, `half_window` | `center`: 1-based int; `half_window`: int radius around center | Local SPC slice | Tuple: `positions = (window,)` int array, `spc = (window,)` float array |
| `boundary_local_spc` | Boundary-centered SPC averaged across all boundaries | `recall_sims`, `N`, `boundary_positions`, `half_window` | `boundary_positions`: list of 1-based ints; `half_window`: int window radius around each boundary | Relative-position SPC summary | Tuple: `rel_positions`, `mean_spc`, `se_spc`; each has shape `(2·half_window+1,)` |
| `summarize_condition_metrics` | One-stop summary of all main metrics for a condition | `recall_sims`, `N`, `boundary_positions`, `half_window` | Same recall matrix; boundary list and SPC window size | Metric bundle | `dict` containing scalar, vector, tuple, and `TransitionResult`-style outputs for the condition |

## `compute_spc`

**Implementation span:** `L52–L57`

### Pseudocode
- Initialize an array of recall probabilities, one slot per serial position.
- For each serial position `j`, check across simulations whether `j` appeared anywhere in the recall sequence.
- Store the mean of that boolean indicator as the SPC value for `j`.
- Return the full SPC vector.

| Pseudocode step | Lines | Code |
|---|---|---|
| Initialize output vector | `L54` | `spc = np.zeros(N)` |
| Loop over serial positions 1…N | `L55` | `for j in range(1, N + 1):` |
| Compute mean recall presence for each position | `L56` | `spc[j - 1] = np.mean(np.any(recall_sims == j, axis=0))` |
| Return SPC | `L57` | `return spc` |

## `compute_pfr`

**Implementation span:** `L60–L68`

### Pseudocode
- Take the first recalled item from each simulation.
- Drop zero entries so only valid first recalls remain.
- Initialize a probability vector over serial positions.
- For each serial position, compute the proportion of trials where that item was recalled first.
- Return the PFR vector.

| Pseudocode step | Lines | Code |
|---|---|---|
| Extract first recall per trial | `L62` | `first = recall_sims[0, :]` |
| Remove zero / empty recalls | `L63` | `first = first[first > 0]` |
| Initialize PFR vector | `L64` | `pfr = np.zeros(N)` |
| Guard against empty first-recall set | `L65` | `if len(first) > 0:` |
| Loop over serial positions | `L66` | `for j in range(1, N + 1):` |
| Compute probability of first recall | `L67` | `pfr[j - 1] = np.mean(first == j)` |
| Return PFR | `L68` | `return pfr` |

## `compute_lag_crp`

**Implementation span:** `L71–L96`

### Pseudocode
- Build the full lag axis from `-(N-1)` to `+(N-1)`.
- Initialize numerator and denominator counts for each lag.
- For each simulation, clean the recall sequence by removing zero entries.
- Walk through consecutive recall transitions.
- At each transition, define the set of still-available items and add one opportunity count to every lag those items would produce from the current item.
- Add one observed count to the actual lag taken by the next recall.
- Convert counts to probabilities wherever the denominator is positive.
- Return the lag axis and CRP values.

| Pseudocode step | Lines | Code |
|---|---|---|
| Create lag axis | `L73–L74` | `max_lag = N - 1; lag_vals = np.arange(-max_lag, max_lag + 1)` |
| Initialize numerator / denominator | `L75–L76` | `numer = np.zeros(…)` / `denom = np.zeros(…)` |
| Map lag to array index | `L77` | `lag_to_idx = {L: i for i, L in enumerate(lag_vals)}` |
| Loop over simulations; clean sequence | `L79–L83` | `for s in …: seq = …; seq = seq[seq > 0]…` |
| Track recalled items | `L84` | `recalled = set()` |
| Loop over transitions; tally opportunities and observed lag | `L85–L91` | `for t in …: recalled.add(cur); remaining = …; denom[…] += 1; numer[…] += 1` |
| Compute CRP only where opportunities exist | `L93–L95` | `crp = np.zeros_like(numer); valid = denom > 0; crp[valid] = numer[valid] / denom[valid]` |
| Return lag axis and CRP | `L96` | `return lag_vals, crp` |

## `recall_accuracy`

**Implementation span:** `L99–L113`

### Pseudocode
- Return `NaN` if the recall matrix is missing or empty.
- For each simulation, remove zero entries from the recall sequence.
- If nothing was recalled, record accuracy as zero.
- Optionally collapse repeated recalls to unique recalled items.
- Compute recalled-items fraction as `len(recalled_items) / N`.
- Average that fraction across simulations.

| Pseudocode step | Lines | Code |
|---|---|---|
| Guard against empty input | `L101–L102` | `if recall_sims is None or recall_sims.size == 0: return np.nan` |
| Initialize per-trial accuracy list | `L103` | `acc = []` |
| Loop over simulations | `L104` | `for s in range(recall_sims.shape[1]):` |
| Remove zero entries | `L105–L106` | `seq = recall_sims[:, s]; seq = seq[seq > 0].astype(int)` |
| Handle empty-recall trial | `L107–L109` | `if seq.size == 0: acc.append(0.0); continue` |
| Optionally unique the recalled items | `L110–L111` | `if unique: seq = np.unique(seq)` |
| Compute per-trial fraction recalled | `L112` | `acc.append(len(seq) / float(N))` |
| Return mean accuracy | `L113` | `return float(np.mean(acc)) if acc else np.nan` |

## `position_conditional_crp`

**Implementation span:** `L127–L182`

### Pseudocode
- Translate the requested lag into an absolute target position.
- Return `NaN` or zero-count result if the target would fall outside list bounds.
- Initialize numerator and denominator.
- For each simulation, remove zero recalls and skip sequences shorter than two recalls.
- Walk through the recall sequence transition by transition while tracking previously recalled items.
- Whenever the current recall equals the requested start position and the target has not yet been recalled, add one to the denominator.
- If the next recall is exactly the target item, also add one to the numerator.
- Return either the probability alone or the probability plus raw counts.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute absolute target position | `L152` | `target_pos = start_pos + target_lag` |
| Guard against out-of-range targets | `L153–L156` | `if target_pos < 1 or target_pos > N: … return np.nan` |
| Initialize counts | `L158–L159` | `num = 0; den = 0` |
| Loop over simulations; clean sequence | `L161–L165` | `for s in …: seq = …; if len(seq) < 2: continue` |
| Track recalled items per transition | `L167–L171` | `recalled = set(); for t in …: cur = seq[t]; nxt = seq[t+1]; recalled.add(cur)` |
| Count opportunities and hits at start position | `L173–L177` | `if cur == start_pos: if target_pos not in recalled: den += 1; if nxt == target_pos: num += 1` |
| Compute probability | `L179` | `prob = (num / den) if den > 0 else np.nan` |
| Return TransitionResult or probability | `L180–L182` | `if return_counts: return TransitionResult(prob, num, den); return prob` |

## `boundary_transition`

**Implementation span:** `L189–L264`

### Pseudocode
- Initialize pooled numerator and denominator.
- For each boundary position `j`, compute the absolute start and target positions from the given offsets.
- Skip any pair where either position falls outside the list.
- Compute the lag between start and target, then call the low-level opportunity-corrected CRP engine to get raw counts.
- Pool those counts across all boundaries.
- Return the pooled probability, optionally with counts.

| Pseudocode step | Lines | Code |
|---|---|---|
| Initialize pooled counts | `L241–L242` | `total_num = 0; total_den = 0` |
| Loop over boundary positions | `L244` | `for j in boundary_positions:` |
| Compute absolute start and target | `L245–L246` | `start = j + start_offset; target = j + target_offset` |
| Skip out-of-range positions | `L249–L252` | `if start < 1 or start > N: continue; if target < 1 or target > N: continue` |
| Compute lag and call low-level CRP | `L254–L257` | `lag = target - start; tr = position_conditional_crp(…, return_counts=True)` |
| Accumulate raw counts | `L258–L259` | `total_num += tr.num; total_den += tr.den` |
| Compute pooled probability | `L261` | `prob = (total_num / total_den) if total_den > 0 else np.nan` |
| Return TransitionResult or probability | `L262–L264` | `if return_counts: return TransitionResult(…); return prob` |

## `local_spc`

**Implementation span:** `L271–L289`

### Pseudocode
- Define a local window around a chosen center position.
- Clip the window so it stays inside the list.
- Compute the full-list SPC.
- Return only the positions and SPC values inside the chosen local window.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute lower / upper bounds of window | `L285–L286` | `lo = max(1, center - half_window); hi = min(N, center + half_window)` |
| Build 1-based position range | `L287` | `pos_range = np.arange(lo, hi + 1)` |
| Compute full SPC | `L288` | `spc_full = compute_spc(recall_sims, N)` |
| Return local positions and local SPC slice | `L289` | `return pos_range, spc_full[pos_range - 1]` |

## `boundary_local_spc`

**Implementation span:** `L292–L325`

### Pseudocode
- Compute whole-list SPC once.
- Create a relative-position axis centered on the boundary.
- For each boundary, convert those relative positions into absolute serial positions.
- Mask positions that would fall outside the list and fill them with `NaN`.
- Extract the SPC values for valid positions to form one local SPC curve per boundary.
- Average local SPC curves across boundaries and compute the standard error across boundaries.
- Return relative positions, mean local SPC, and standard error.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute whole-list SPC | `L309` | `spc_full = compute_spc(recall_sims, N)` |
| Create relative-position axis | `L310` | `rel = np.arange(-half_window, half_window + 1)` |
| Loop over boundaries; extract local curves | `L312–L318` | `for j in …: abs_pos = j + rel; valid = …; curve[valid] = spc_full[…]; curves.append(curve)` |
| Stack curves into array | `L320` | `curves = np.array(curves)` |
| Compute mean across boundaries | `L321` | `mean_spc = np.nanmean(curves, axis=0)` |
| Compute standard error across boundaries | `L322–L324` | `se_spc = np.nanstd(…) / np.sqrt(np.sum(~np.isnan(…), axis=0))` |
| Return relative positions, mean, SE | `L325` | `return rel, mean_spc, se_spc` |

## `summarize_condition_metrics`

**Implementation span:** `L332–L366`

### Pseudocode
- Compute whole-list lag-CRP.
- Compute boundary-local SPC summary.
- Bundle whole-list metrics, pooled boundary transition metrics, and local SPC into one dictionary.
- Return that dictionary as a compact summary for one condition.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute whole-list lag-CRP | `L348` | `lag_vals, crp = compute_lag_crp(recall_sims, N)` |
| Compute boundary-local SPC | `L349–L351` | `rel, mean_spc_local, se_spc_local = boundary_local_spc(…)` |
| Build summary dictionary | `L353–L366` | `return {"recall_accuracy": …, "whole_spc": …, "whole_pfr": …, …}` |

# `simulation.py`

## Function summary

| Function | Purpose | Key inputs | Input structure | Output | Output structure |
|---|---|---|---|---|---|
| `build_boundary_schedule` | Construct an H1 drift schedule with fixed non-boundary drift and additive boundary boost | `B_non`, `delta`, `boundary_positions` | `B_non`: float baseline drift for non-boundary items; `delta`: float additive boost at boundaries; `boundary_positions`: list or array of 1-based boundary positions | Boundary-boosted drift schedule | `(N,)` float array of per-position drift values, clipped to `[0, 1]` |
| `build_global_schedule` | Construct an H2 drift schedule with globally shifted non-boundary drift and fixed boundary drift | `B_non`, `B_boundary`, `boundary_positions` | `B_non`: float drift for non-boundary items; `B_boundary`: float drift for boundary items; `boundary_positions`: list or array of 1-based boundary positions | Global-tonic drift schedule | `(N,)` float array of per-position drift values, clipped to `[0, 1]` |
| `build_baseline_schedule` | Construct the default healthy baseline drift schedule from config defaults | `boundary_positions` | `boundary_positions`: list or array of 1-based boundary positions | Baseline drift schedule | `(N,)` float array of per-position drift values |
| `describe_schedule` | Give a short human-readable summary of a drift schedule | `B_encD`, `boundary_positions` | `B_encD`: `(N,)` float array of drift values; `boundary_positions`: list or array of 1-based boundary positions | Schedule description | Multi-line string summarizing list length, boundary locations, mean/range of non-boundary drift, and mean/range of boundary drift |
| `_dot` | Compute scalar dot product of two vectors / column arrays | `a`, `b` | Array-like vectors or column arrays with compatible shapes | Scalar dot product | `float` |
| `_norm` | Compute Euclidean norm of a vector | `v` | Array-like vector | Vector norm | `float` |
| `simulate_single_trial` | Run one full CMR encode→retrieve trial under a specified drift schedule | `B_encD`, `rng`, `gamma_fc`, `eta`, `B_rec` | `B_encD`: `(N,)` float array of per-position encoding drift rates; `rng`: NumPy random generator; optional overrides: `gamma_fc`, `eta`, `B_rec` as floats or `None` | One trial’s recall outputs and final matrices | Tuple: `recalls = (N,)` int array of 1-based recalled serial positions with `0` padding; `times = (N,)` float array of cumulative recall times; `net_w_fc = (N, N)` float matrix; `net_w_cf = (N, N)` float matrix |
| `run_batch` | Run many CMR trials under one fixed drift schedule and collect batch outputs | `B_encD`, `n_sims`, `seed`, `label`, `sim_kwargs` | `B_encD`: `(N,)` float array; `n_sims`: int number of trials; `seed`: int RNG seed; `label`: string condition name; `sim_kwargs`: optional parameter overrides forwarded to `simulate_single_trial` | Batch simulation bundle | `dict` with keys: `label` (str), `B_encD` (`(N,)` float array), `recall_sims` (`(N, n_sims)` int array), `times_sims` (`(N, n_sims)` float array), `net_w_fc` (`(N, N)` float matrix), `net_w_cf` (`(N, N)` float matrix) |

## `build_boundary_schedule`

**Implementation span:** `L31–L58`

### Pseudocode
- Create a length-`N` vector filled with the non-boundary baseline drift.
- For every boundary position, replace the corresponding entry with `B_non + delta`.
- Clip the resulting schedule to `[0, 1]` and return it.

| Pseudocode step | Lines | Code |
|---|---|---|
| Initialize all positions to baseline drift | `L55` | `B = np.full(N, B_non, dtype=float)` |
| Overwrite boundary positions with boosted drift | `L56–L57` | `for j in boundary_positions: B[j - 1] = B_non + delta` |
| Clip and return schedule | `L58` | `return np.clip(B, 0.0, 1.0)` |

## `build_global_schedule`

**Implementation span:** `L61–L88`

### Pseudocode
- Create a length-`N` vector filled with the chosen non-boundary drift.
- For every boundary position, replace the entry with the chosen boundary drift.
- Clip to `[0, 1]` and return.

| Pseudocode step | Lines | Code |
|---|---|---|
| Initialize all positions to non-boundary drift | `L85` | `B = np.full(N, B_non, dtype=float)` |
| Overwrite all boundary positions | `L86–L87` | `for j in boundary_positions: B[j - 1] = B_boundary` |
| Clip and return schedule | `L88` | `return np.clip(B, 0.0, 1.0)` |

## `build_baseline_schedule`

**Implementation span:** `L91–L97`

### Pseudocode
- Call the boundary-schedule constructor using configuration defaults for the baseline non-boundary drift and baseline boundary boost.
- Return the resulting baseline schedule.

| Pseudocode step | Lines | Code |
|---|---|---|
| Delegate to build_boundary_schedule with config defaults | `L95–L97` | `return build_boundary_schedule(B_NON_BOUNDARY_BASE, B_BOUNDARY_DELTA_BASE, boundary_positions)` |

## `describe_schedule`

**Implementation span:** `L100–L110`

### Pseudocode
- Convert the boundary list into a set for membership checks.
- Split the drift vector into non-boundary values and boundary values.
- Compute summary statistics for each group.
- Format those statistics into a short human-readable multi-line string.

| Pseudocode step | Lines | Code |
|---|---|---|
| Create set of boundary positions | `L102` | `bps = set(boundary_positions)` |
| Collect non-boundary drift values | `L103` | `non_vals = [B_encD[i] for i in range(len(B_encD)) if (i + 1) not in bps]` |
| Collect boundary drift values | `L104` | `bdy_vals = [B_encD[j - 1] for j in boundary_positions]` |
| Assemble summary strings | `L105–L109` | `parts = [f"N = {len(B_encD)}, …", f"non-boundary drift: …", f"boundary drift: …"]` |
| Join and return description | `L110` | `return "\n".join(parts)` |

## `_dot`

**Implementation span:** `L117–L118`

### Pseudocode
- Compute the scalar dot product of two vectors / column arrays and return it as a float.

| Pseudocode step | Lines | Code |
|---|---|---|
| Dot product and scalar conversion | `L118` | `return float((np.asarray(a).T @ np.asarray(b)).ravel()[0])` |

## `_norm`

**Implementation span:** `L121–L122`

### Pseudocode
- Compute the Euclidean norm of a vector and return it as a float.

| Pseudocode step | Lines | Code |
|---|---|---|
| Norm and scalar conversion | `L122` | `return float(np.linalg.norm(v))` |

## `simulate_single_trial`

**Implementation span:** `L129–L258`

### Pseudocode
- Read default parameters from configuration and optionally override selected ones.
- Precompute encoding and retrieval constants, and cast the drift schedule to a float array.
- Initialize item features, context state, and the two associative weight matrices.
- Encoding phase: for each study position, activate the studied item feature, compute incoming context, update context using the drift value for that position, and write new associations into `M_FC` and `M_CF`.
- Initialize recall outputs, retrieved-item mask, thresholds, and the mixed episodic/semantic retrieval weights.
- Retrieval phase: repeatedly accumulate evidence over short cycles using competition and noise until an item crosses threshold or recall time expires.
- When an item wins, map the feature index back to serial position, update context using recall drift, update the retrieval matrices if enabled, record the recalled serial position and time, and mark the item as retrieved.
- Continue until recall time runs out, then return recalled positions, times, and final weight matrices.

| Pseudocode step | Lines | Code |
|---|---|---|
| Read / override parameters | `L155–L173` | `p = BASE_PARAMS; gamma_fc = … if … else p["gamma_fc"]; …; B_encD = np.asarray(B_encD, dtype=float)` |
| Initialize context, features, and matrices | `L175–L178` | `net_f = np.zeros((N,1)); net_c = np.zeros((N,1)); net_w_fc = np.eye(N)*eye_fc; net_w_cf = np.zeros((N,N))` |
| Encoding loop | `L181–L195` | `for pos in range(N): … net_c = rho*net_c + B*net_c_in; net_w_fc += …; net_w_cf += …` |
| Initialize retrieval state | `L198–L205` | `recalls = np.zeros(N, dtype=int); … net_weights = episodic_w*net_w_cf + sem_w*sem_mat; …` |
| Outer retrieval loop over remaining recall time | `L207–L235` | `while time_passed < rec_time: f_in = net_weights @ net_c; … time_passed += i*dt` |
| Threshold crossing / winner selection | `L222–L233` | `while i < max_cycles and not crossed: x = x + …; … if np.any(x[retrievable] >= thresholds[…]): crossed = True; …` |
| Process winning recall and update state | `L237–L256` | `if crossed and winners is not None: winner = …; net_c = rho*net_c + B_rec*net_c_in; recalls[…] = sp1; …` |
| Return recalls, times, and final matrices | `L258` | `return recalls, times, net_w_fc, net_w_cf` |

## `run_batch`

**Implementation span:** `L265–L299`

### Pseudocode
- Initialize a random-number generator and cast the drift schedule to an array.
- Allocate arrays to store recalled positions and recall times across simulations.
- Loop over simulations and run one trial each time using the same drift schedule.
- Store recall outputs and keep the final weight matrices from the last run.
- Return all stored results in a dictionary.

| Pseudocode step | Lines | Code |
|---|---|---|
| Initialize RNG and drift schedule array | `L279–L280` | `rng = np.random.default_rng(seed); B = np.asarray(B_encD, dtype=float)` |
| Allocate batch output arrays | `L281–L284` | `recall_sims = np.zeros((N, n_sims), dtype=int); times_sims = …; wfc_last = None; wcf_last = None` |
| Loop over simulations | `L286–L290` | `for s in range(n_sims): rec, t, wfc, wcf = simulate_single_trial(B, rng, **sim_kwargs); …` |
| Return results dictionary | `L292–L299` | `return {"label": label, "B_encD": B, "recall_sims": …, "times_sims": …, …}` |